Introduction
============

This is an interactive worksheet for "Introduction to path planning" (Author: Björn Hein).

Version | Author
------------ | -------------
0.2 | Björn Hein


License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (please check: http://creativecommons.org/licenses/by-nc/4.0/)

This worksheet contains a second draft implementation of the A* algorithm for path planning applications in Python. It follows the slides "A* - Algorithm for path planning": discretisation of the C-space, the A* algorithm, heuristics, the heuristic weight and reopening. At the end it shows the effect of the step sizes of the discretisation.

Important links are:

* General Info: http://www.python.org
* Python tutorial http://www.python.org/doc/tut/
* NetworkX http://networkx.github.io/
* NumPy and SciPy Guide http://docs.scipy.org/
* Matplotlib gallery http://matplotlib.sourceforge.net/gallery.html


Remember that:

* you have to press ctrl-return or shift-return, to execute the code in the code sections, only then the variables are "generated" and can be used
* you can execute the whole notebook by Cell->runAll

Discretisation of C-space
=========================

To use A* as path planning algorithm, the C-space must be discretised: it becomes a grid on which A* can "take place". Every grid point corresponds to exactly one configuration.

* Joint $i$: limits $\Theta_{\min,i} \dots \Theta_{\max,i}$, step size $\Delta\Theta_i$
* Grid coordinate $x_i \in \{0, 1, \dots, N_i - 1\}$ with $N_i = \frac{\Theta_{\max,i} - \Theta_{\min,i}}{\Delta\Theta_i} + 1$
* Grid point → configuration: $\varphi_i = \Theta_{\min,i} + \Delta\Theta_i\, x_i$
* Configuration → grid point: $x_i = \left\lfloor \frac{\varphi_i - \Theta_{\min,i}}{\Delta\Theta_i} + \frac{1}{2} \right\rfloor$

The example with two joints from the slides:

| Joint $i$ | $\Theta_{\min,i}$ | $\Theta_{\max,i}$ | $\Delta\Theta_i$ | $N_i$ |
| :-------: | ----------------: | ----------------: | ---------------: | ----: |
| 1         |              −90° |               90° |              20° |    10 |
| 2         |                0° |              135° |              15° |    10 |

In [ ]:
import math


def toConfiguration(x, thetaMin, deltaTheta):
    """Grid point -> configuration: phi_i = thetaMin_i + deltaTheta_i * x_i"""
    return [tMin + d * xi for xi, tMin, d in zip(x, thetaMin, deltaTheta)]


def toGrid(phi, thetaMin, deltaTheta):
    """Configuration -> grid point: x_i = floor((phi_i - thetaMin_i) / deltaTheta_i + 1/2)"""
    return [math.floor((p - tMin) / d + 0.5) for p, tMin, d in zip(phi, thetaMin, deltaTheta)]


thetaMin = [-90, 0]
thetaMax = [90, 135]
deltaTheta = [20, 15]

N = [int((tMax - tMin) / d) + 1 for tMin, tMax, d in zip(thetaMin, thetaMax, deltaTheta)]
print("N =", N)
print("grid point x = (5, 2)          -> configuration", toConfiguration([5, 2], thetaMin, deltaTheta))
x = toGrid([45, 50], thetaMin, deltaTheta)
print("configuration phi = (45°, 50°) -> grid point", x, "=", toConfiguration(x, thetaMin, deltaTheta))

### Grid coordinates: why + ½?

Joint 1 and the configuration $\varphi_1 = 45°$: $\frac{45° + 90°}{20°} = 6.75$, the configuration lies **between** grid points 6 and 7.

* **Without** $+\frac{1}{2}$ the grid point covers the range **above** it: $\lfloor 6.75 \rfloor = 6$ → 30°, 15° away.
* **With** $+\frac{1}{2}$ the grid point covers the range **around** it (its field): $\lfloor 6.75 + 0.5 \rfloor = 7$ → 50°, 5° away. The error is at most $\frac{\Delta\Theta_i}{2}$.

Python's `round()` is no replacement: it rounds 6.5 to 6 (half to even).

In [ ]:
for phi in [45, 30, 39, 41]:
    value = (phi - thetaMin[0]) / deltaTheta[0]
    without, withHalf = math.floor(value), math.floor(value + 0.5)
    print(f"phi = {phi}°: {value:.2f} -> without + 1/2: x = {without} ({toConfiguration([without], thetaMin, deltaTheta)[0]}°), "
          f"with + 1/2: x = {withHalf} ({toConfiguration([withHalf], thetaMin, deltaTheta)[0]}°)")

Calculating the hash value for nodes needed for the closed list
====================================

As the closed list of the $A^*$ algorithm is typically implemented via a *hash map*, an appropriate hash function has to be provided to identify nodes. On the slides it is the `tHashtable` CLOSED. A suitable key is based on the grid coordinates $x$ of the nodes: every grid point corresponds to exactly one configuration, and integer coordinates are exact (with floating point numbers e.g. `0.1 + 0.2 != 0.3`).

In [ ]:
def getNodeID(x):
    """Compute a unique identifier based on the grid coordinates"""
    nodeId = "-"
    for i in x:
        nodeId += str(i)+"-"
    return nodeId

In [ ]:
getNodeID([20,20])

In [ ]:
a = dict()
a[getNodeID([20,20])] = 99
a[getNodeID([31,30])] = 101

print(a[getNodeID([31,30])], getNodeID([31,30]))
print(a[getNodeID([20,20])])


A*
==

In the following there is an example for programming an A*-like path planning algorithm with some modifications. Please check the slides "A* - Algorithm for path planning". The pseudocode of the slides in `IPAStar.py`:

| Slides | IPAStar.py |
| ------ | ---------- |
| `init (OPEN; CLOSED; S)` | `planPath`: reset, `_addGraphNode` for the grid point of the start |
| `n = best (OPEN)` | `_getBestNodeName`: `heapq` sorted by $f$ from `_evaluateNode` |
| `n == G` | the name of `n` is the name of the grid point of the goal |
| collision check | `pointInCollision`, when `n` is taken from OPEN: collisions are only detected then |
| `expandNode (n)` and `handleSuccessors (OPEN; CLOSED; s)` | merged to one function `_handleNode`: neighbours $x_i \pm 1$, no reopening |
| `printSolution (…)` | `_collectPath`: follows the fathers from G back to S |
| OPEN (`tHeap`), CLOSED (`tHashtable`) | `self.openList` (`heapq`), `self.graph` (`networkx.DiGraph`, key from `_getNodeID`) |

In [ ]:
from IPAStar import AStar
from IPVISAStar import aStarVisualize
import matplotlib.pylab as plt

In [ ]:
import IPTestSuite as ts

In [ ]:
astarConfig = dict()
astarConfig["w"] = .5
astarConfig["heuristic"]  = "euclidean"

for benchmark in ts.benchList:
    try:
        fig_local = plt.figure(figsize=(5,5))
        ax = fig_local.add_subplot(1,1,1)
        astar = AStar(benchmark.collisionChecker)
        solution = astar.planPath(benchmark.startList, benchmark.goalList, astarConfig)
        title = benchmark.name
        if not solution:
            title += " (No path found!)"
        title += "\n Assumed complexity level " + str(benchmark.level)
        ax.set_title(title)
        aStarVisualize(astar, solution, ax=ax, nodeSize=50)
    except Exception as e:
        print("ERROR: ",benchmark.name, e)

Questions:

* What could be useful to be checked in function setStartGoal besides start and goal having the same dimension?
* What is missing for a correct implementation of the A*? 
  * Why does this version not necessarily find the best solution? 
  * Why is it nevertheless useful to do so?



### Do the planning

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

from shapely.geometry import Point, Polygon, LineString
from IPEnvironment import CollisionChecker

# Definition of a testing environment
trapField = dict()
trapField["obs1"] =   LineString([(6, 18), (6, 8), (16, 8), (16,18)]).buffer(1.0)

environment = CollisionChecker(trapField)


start = [[10,20]]
goal  = [[13,1]]

# AStar
astar = AStar(environment)

# To know what is part on the config please consult the documentation of planPath
astarConfig = dict()
astarConfig["w"] = .5

astarConfig["heuristic"]  = "euclidean"


solution = astar.planPath(start, goal, astarConfig )


### Do the visualization

* Idea: Show internal data of AStar
* show closeList and the evaluated values for these nodes
* add these information in a matrix

In [ ]:
fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(1,1,1)
aStarVisualize(astar, solution, ax = ax, nodeSize=50)

Comparison: used metrics
========================

The trap example from above with $w = 0.5$, once with the Manhattan and once with the Euclidean distance as heuristic. The numbers are the same as on the slides: closed nodes, nodes in collision and edges of the path.

In [ ]:
import numpy as np
from scipy.spatial.distance import euclidean


def statistics(planner, solution):
    """Closed nodes, nodes in collision, reopenings and updates in OPEN, edges and length of the path"""
    nodes = planner.graph.nodes(data=True)
    result = {"closed": sum(1 for _, data in nodes if data["status"] == "closed"),
              "in collision": sum(1 for _, data in nodes if data.get("collision") == 1),
              "reopened": sum(data.get("reopened", 0) for _, data in nodes),
              "updated": sum(data.get("updated", 0) for _, data in nodes),
              "edges": None, "length": None}
    if solution:
        points = [planner.graph.nodes[node]["pos"] for node in solution]
        result["edges"] = len(solution) - 1
        result["length"] = sum(euclidean(p, q) for p, q in zip(points, points[1:]))
    return result


def planAndShow(collisionChecker, startList, goalList, runs, panelSize=5):
    """Plan once for every (title, config) in runs and draw the results side by side.
    config completes the base configuration w = 0.5, heuristic "euclidean"."""
    fig, axes = plt.subplots(1, len(runs), figsize=(panelSize * len(runs), panelSize + 0.6))
    limits = collisionChecker.getEnvironmentLimits()
    for ax, (title, config) in zip(np.atleast_1d(axes), runs):
        config = {"w": 0.5, "heuristic": "euclidean", **config}
        astar = AStar(collisionChecker)
        solution = astar.planPath(startList, goalList, config)
        result = statistics(astar, solution)
        title += f"\nclosed: {result['closed']} · in collision: {result['in collision']}"
        title += f"\npath: {result['edges']} edges, length {result['length']:.1f}" if solution else "\nno path found!"
        if config.get("reopening"):
            title += f"\nreopenings: {result['reopened']} · updates in OPEN: {result['updated']}"
        ax.set_title(title)
        # the size of the nodes follows the smallest step size
        step = min(np.atleast_1d(config.get("discretization", 1)))
        aStarVisualize(astar, solution, ax=ax, nodeSize=min((1.6 * panelSize * step)**2, 300))
        ax.set_xlim(limits[0][0] - 0.5, limits[0][1] + 0.5)
        ax.set_ylim(limits[1][0] - 0.5, limits[1][1] + 0.5)
        ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

In [ ]:
planAndShow(environment, start, goal, [("Manhattan", {"heuristic": "manhattan"}),
                                       ("Euclidean", {"heuristic": "euclidean"})])

Modifying evaluation function
=============================

Modified evaluation function: $f(n) = (1-w)\,g(n) + w\,h(n)$

* $w$: heuristic weight · $w = 0$: path costs only · $w = 0.5$: classic A* · $w = 1$: heuristic only

In [ ]:
planAndShow(environment, start, goal, [(f"w = {w}", {"w": w}) for w in [0, 0.5, 1]])

### Heuristic weight: trap example

$w$ from 0 to 1 in steps of 0.05, with both heuristics:

In [ ]:
weights = [i / 20 for i in range(21)]
fig, (axClosed, axEdges) = plt.subplots(1, 2, figsize=(12, 4))
for heuristic, label, color, marker in [("manhattan", "Manhattan", "#2a78d6", "o"),
                                        ("euclidean", "Euclidean", "#eda100", "s")]:
    results = []
    for w in weights:
        astar = AStar(environment)
        results.append(statistics(astar, astar.planPath(start, goal, {"w": w, "heuristic": heuristic})))
    axClosed.plot(weights, [r["closed"] for r in results], color=color, marker=marker, lw=2, ms=6, label=label)
    axEdges.plot(weights, [r["edges"] for r in results], color=color, marker=marker, lw=2, ms=6, label=label)
for ax, ylabel in [(axClosed, "closed nodes"), (axEdges, "path length (edges)")]:
    ax.set(xlabel="w", ylabel=ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

Reopening
=========

`handleSuccessors` on the slides: what happens, if a successor $s'$ is already in OPEN or CLOSED?

* **Without reopening** (default of `IPAStar`): $s'$ is skipped, the first path to a node is kept.
* **With reopening** (`astarConfig["reopening"] = True`): if the new path is cheaper, $s'$ gets the new father and the lower $g$ and goes (back) into OPEN. A node in OPEN gets a better place in the heap, a closed node is reopened and corrects its successors, when it is expanded again.

```
procedure handleSuccessors (OPEN; CLOSED; s)
begin
    for each s' in s do begin
        g' = g(n) + c(n, s');
        if (s' ∉ OPEN ∪ CLOSED) then
            father (s') = n;  g(s') = g';  insert (OPEN; s');
        else if (g' < g(s')) then          // cheaper path found
            father (s') = n;  g(s') = g';
            if (s' ∈ CLOSED) then
                move (s'; CLOSED; OPEN);   // reopening
            else
                update (OPEN; s');         // new f(s')
            endif
        endif
    end
end;
```

In the figures nodes, that got a cheaper path while in OPEN, have a violet ring, reopened nodes are violet.

### Classic A* (w = 0.5)

A bar on a 10 x 10 grid, Manhattan distance:

In [ ]:
bar = CollisionChecker({"bar": Polygon([(3.5, 4.5), (8.5, 4.5), (8.5, 5.5), (3.5, 5.5)])}, limits=[[0, 9], [0, 9]])
barStart, barGoal = [[8, 9]], [[7, 4]]
planAndShow(bar, barStart, barGoal, [("without reopening", {"heuristic": "manhattan"}),
                                     ("with reopening", {"heuristic": "manhattan", "reopening": True})])

for reopening in [False, True]:
    astar = AStar(bar)
    astar.planPath(barStart, barGoal, {"w": 0.5, "heuristic": "manhattan", "reopening": reopening})
    g = {tuple(data["pos"]): data["g"] for _, data in astar.graph.nodes(data=True)}
    print(f"reopening {reopening}: g of the nodes below S:", [g[(8, y)] for y in [8, 7, 6]])

Without reopening the nodes below S are first reached over the detour on the left (with equal $f$ the order of the names in the heap decides), the cheaper path found later is skipped. With reopening they are updated in OPEN. The heuristic is consistent for $w \le 0.5$: closed nodes are never reopened.

### Weighted A* (w = 1)

For $w > 0.5$ the heuristic weighs more than the path costs: nodes are closed before their cheapest path is known.

In [ ]:
planAndShow(environment, start, goal, [("w = 1, without reopening", {"w": 1}),
                                       ("w = 1, with reopening", {"w": 1, "reopening": True})])

### Reopening and the heuristic weight

The trap example, $w$ from 0 to 1 in steps of 0.05:

In [ ]:
fig, (axEdges, axReopened) = plt.subplots(1, 2, figsize=(12, 4))
for heuristic, label, color, marker in [("manhattan", "Manhattan", "#2a78d6", "o"),
                                        ("euclidean", "Euclidean", "#eda100", "s")]:
    for reopening, style in [(False, "-"), (True, "--")]:
        results = []
        for w in weights:
            astar = AStar(environment)
            solution = astar.planPath(start, goal, {"w": w, "heuristic": heuristic, "reopening": reopening})
            results.append(statistics(astar, solution))
        name = label + (" + reopening" if reopening else "")
        axEdges.plot(weights, [r["edges"] for r in results], style, color=color, marker=marker, lw=2, ms=6, label=name)
        if reopening:
            axReopened.plot(weights, [r["reopened"] for r in results], style, color=color, marker=marker, lw=2, ms=6,
                            label=label)
for ax, ylabel in [(axEdges, "path length (edges)"), (axReopened, "reopenings (closed → OPEN)")]:
    ax.set(xlabel="w", ylabel=ylabel)
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

### All benchmarks

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for benchmark in ts.benchList:
    for heuristic in ["manhattan", "euclidean"]:
        for w in [0.5, 0.7, 1]:
            row = {"benchmark": benchmark.name, "heuristic": heuristic, "w": w}
            for reopening in [False, True]:
                astar = AStar(benchmark.collisionChecker)
                solution = astar.planPath(benchmark.startList, benchmark.goalList,
                                          {"w": w, "heuristic": heuristic, "reopening": reopening})
                result = statistics(astar, solution)
                suffix = " (reopening)" if reopening else ""
                row["edges" + suffix] = result["edges"]
                row["closed" + suffix] = result["closed"]
            row["reopenings"], row["updates in OPEN"] = result["reopened"], result["updated"]
            rows.append(row)
display(pd.DataFrame(rows))

Questions:

* Why is no closed node reopened for $w \le 0.5$? (Hint: consistent heuristic)
* In the bar example the heuristic is consistent, but without reopening the path is longer. Which part of `handleSuccessors` does the classic A* need?
* For which benchmarks and $w$ is the path **longer** with reopening? How can that be?
* What does reopening cost?

Experiments
===========

* Implement another heuristic (e.g. the maximum norm $\max_i |x_i - x_{G,i}|$) and make it switchable via the parameter `heuristic`. Which of the heuristics never overestimate the path costs in the 4-neighbourhood?
* Use the diagonal neighbours as well (8-neighbourhood, see `_handleNode9`). Which costs should a diagonal step get, and which heuristic fits then?


Discretisation per dimension in IPAStar
=======================================

`IPAStar` discretises the C-space as on the slides (see the beginning of this worksheet): $\Theta_{\min,i}$ are the lower limits of the collision checker (`getEnvironmentLimits()`), the step sizes $\Delta\Theta_i$ are part of the configuration:

    astarConfig["discretization"] = [1.0, 0.5]   # step size per dimension, a single number for all, default: 1
    astarConfig["maxIterations"]  = 20000        # a finer grid needs more expansions, default: 1000

* A* takes place on the grid: the successors of a node are its neighbours $x_i \pm 1$, every edge costs 1 ($g$ counts the edges) and the heuristic measures the distance in grid coordinates.
* The configuration $\varphi_i = \Theta_{\min,i} + \Delta\Theta_i\, x_i$ of a node is needed for the collision check and for the path.
* Start and goal belong to the grid point of their field, $x_i = \left\lfloor \frac{\varphi_i - \Theta_{\min,i}}{\Delta\Theta_i} + \frac{1}{2} \right\rfloor$. Their nodes keep the exact configuration, so that the path starts in S and ends in G; `n == G` means: `n` is the grid point of the goal.
* With $\Delta\Theta_i = 1$ and $\Theta_{\min,i} = 0$, as in all benchmarks, grid coordinates and configurations are the same: the results above stay as they were.

### Same step size in all dimensions

The trap example with a coarser, the original and a finer discretisation:

In [ ]:
planAndShow(environment, start, goal, [(f"ΔΘ = {d}", {"discretization": d, "maxIterations": 20000}) for d in [2, 1, 0.3]])

### Different step sizes per dimension

Every edge costs 1, whatever the step size: different step sizes make sense when the dimensions have different units or a different effect, e.g. the joint angles of a robot. With $\Delta\Theta = (0.3, 0.7)$ start and goal do not lie on grid points.

In [ ]:
planAndShow(environment, start, goal, [(f"ΔΘ = {d}", {"discretization": d, "maxIterations": 20000})
                                       for d in [[1, 0.5], [0.5, 1], [0.3, 0.7]]])

### Effort and path on all benchmarks

In [ ]:
import pandas as pd
from IPython.display import display

stepSizes = [2, 1, 0.5, 0.25]
rows = []
for benchmark in ts.benchList:
    for d in stepSizes:
        astar = AStar(benchmark.collisionChecker)
        solution = astar.planPath(benchmark.startList, benchmark.goalList,
                                  {"w": 0.5, "heuristic": "euclidean", "discretization": d, "maxIterations": 20000})
        result = statistics(astar, solution)
        rows.append({"benchmark": benchmark.name, "ΔΘ": d, "closed": result["closed"], "edges": result["edges"],
                     "length": round(result["length"], 1) if solution else None})
results = pd.DataFrame(rows)

for value in ["closed", "edges", "length"]:
    print(value)
    display(results.pivot(index="benchmark", columns="ΔΘ", values=value)[stepSizes])

### Too coarse?

The fat bottleneck (the passage between the obstacles is 1 wide, from 11.5 to 12.5) with $\Delta\Theta$ = 4, 2.5 and 2, and a maze (the walls are 0.3 thick) with $\Delta\Theta$ = 1, 0.5 and 0.25:

In [ ]:
from IPMaze import mazeBenchmark

fatBottleneck = next(benchmark for benchmark in ts.benchList if benchmark.name == "Fat bottleneck")
planAndShow(fatBottleneck.collisionChecker, fatBottleneck.startList, fatBottleneck.goalList,
            [(f"ΔΘ = {d}", {"discretization": d, "maxIterations": 20000}) for d in [4, 2.5, 2]])

maze = mazeBenchmark(rows=6, cols=6, seed=3)
planAndShow(maze.collisionChecker, maze.startList, maze.goalList,
            [(f"ΔΘ = {d}", {"discretization": d, "maxIterations": 20000}) for d in [1, 0.5, 0.1]])

Questions:

* How does the number of closed nodes grow when the step size is halved? What do you expect for a robot with 6 axes?
* Why is there no path through the fat bottleneck with $\Delta\Theta = 2.5$, but with 4 and 2? (Hint: which grid points lie in the passage?)
* Why does the path cross the walls of the maze with $\Delta\Theta = 1$? Why does $\Delta\Theta = 0.5$ work here, although the walls are only 0.3 thick – would it still work, if the maze were shifted by 0.25?
* Extend `_handleNode`, so that only collision-free edges are added (`lineInCollision`). What does this cost?
* Every edge costs 1, whatever $\Delta\Theta_i$ is. How would you choose the step sizes for the axes of a robot? (Hint: how far does the tool move, if axis $i$ moves by $\Delta\Theta_i$? See the slides on hierarchical A*.)